# 3.5 — Reducción de dimensionalidad para visualización

**Aspecto del rúbrica:** *Uso de métodos de reducción de características para
visualización (PCA, t-SNE, …)*.

Cada semana tiene 5 variables de clima y aire: temperatura, humedad, sequía,
PM10 y ozono. No se pueden dibujar 5 dimensiones a la vez, así que estas
técnicas las **resumen en 2 ejes** para poder ver todas las semanas en una sola
gráfica y buscar grupos.

Responde tres preguntas:

1. ¿Se pueden resumir las 5 variables en pocas sin perder mucha información?
2. ¿Las semanas forman grupos? ¿Qué los separa?
3. ¿Esos grupos tienen que ver con los casos de IRA, más allá de la época del año?

## Resumen

- **Dos ejes guardan dos terceras partes de la información** (66 %) de las 5
  variables. El primero separa la **primavera seca** (sequía y ozono altos) de
  las semanas **húmedas**; el segundo separa el **invierno frío y con polvo** del
  **verano caluroso**.
- **Las semanas se agrupan por temporada** (frío, seco, monzón), tanto con PCA
  como con t-SNE. Los casos altos caen en el grupo del invierno: otra vez es la
  época del año, como en 3.3 y 3.4.
- **Sin la época, los grupos desaparecen.** El eje principal pasa a ser "semana
  más calurosa, seca y con más polvo de lo normal" (47 %), pero **no se
  relaciona con los casos** (−0.12).
- **Conclusión:** no hay un tipo de semana, por su clima o su aire, que tenga más
  casos de lo normal. Temperatura, humedad y PM10 cuentan casi la misma
  historia y, en un modelo, se podrían resumir en un solo índice.

## Las dos técnicas

- **PCA (análisis de componentes principales):** busca el "ángulo" desde el que
  los datos se ven más extendidos y lo llama *eje 1*; luego el mejor ángulo que
  queda y lo llama *eje 2*. Cada eje es una **mezcla de las variables**, y sus
  *pesos* dicen cuánto aporta cada una: un peso grande (positivo o negativo)
  significa que esa variable define el eje.
- **t-SNE:** acomoda las semanas en un plano de forma que las que se parecen
  queden **juntas**. Sirve para ver grupos, pero las distancias grandes y los
  ejes no tienen un significado directo.

## Cómo se hace

- Como en 3.4: **sin 2020** (pandemia), con **valores medidos** (sin los que
  rellenó 3.1), y sin olas de calor ni marginación (`IM`), que no cambian de
  forma útil semana a semana.
- Estas técnicas necesitan las 5 variables completas, así que se usan solo las
  **225 semanas** de 2015–2019 con todo medido (faltan sobre todo las del
  apagón de la estación en 2018).
- Antes se **estandariza** cada variable (se le resta su promedio y se divide
  entre su variación típica). Sin eso, una variable con números grandes, como el
  PM10, dominaría solo por su escala.

| Entrada | Salida |
|---------|--------|
| `data/processed/panel_eda_semanal.parquet` o `panel_eda.parquet` (según `GRANULARIDAD`), del paso 7 de `3.0` | Figuras `reports/figures/pca_*_{GRANULARIDAD}.png` y `tsne_{GRANULARIDAD}.png` |

## Configuración e imports

Se carga el panel, se deja fuera 2020, se quitan las semanas incompletas y se
prepara la versión **sin la época** (cada valor menos lo normal para su mes),
igual que en 3.4.

In [ ]:
from pathlib import Path  # Path: manejar rutas de carpetas y archivos
import sys  # sys: cambiar dónde busca módulos Python

# Este notebook vive en notebooks/03_eda/: subimos dos carpetas hasta la raíz del proyecto
ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt  # gráficas
import numpy as np  # cálculos numéricos
import pandas as pd  # tablas
import seaborn as sns  # gráficas con colores por grupo y mapas de calor
from sklearn.decomposition import PCA  # PCA
from sklearn.manifold import TSNE  # t-SNE
from sklearn.preprocessing import StandardScaler  # estandarizar variables

from sonora_health_climate_eda.config import FIGURES_DIR, PROCESSED_DATA_DIR

# "semanal" = análisis principal; "mensual" = comparación
GRANULARIDAD = "semanal"

# Nombre del archivo y de las columnas según la granularidad (igual que en 3.2–3.4)
_CFG = {
    "semanal": {"archivo": "panel_eda_semanal.parquet", "fecha": "fecha_semana",
                "y": "casos_ira_total", "tmax": "tmax_media_sem"},
    "mensual": {"archivo": "panel_eda.parquet", "fecha": "fecha_mes",
                "y": "casos_ira_por_semana", "tmax": "tmax_media_mes"},
}[GRANULARIDAD]
FECHA, Y, TMAX = _CFG["fecha"], _CFG["y"], _CFG["tmax"]

# Las 5 variables de clima y aire que se resumen (los casos NO entran: solo se usan
# para colorear las gráficas y ver si coinciden con los grupos)
VARIABLES = [TMAX, "humedad_rel_media", "severidad_sequia", "pm10_ugm3", "o3_max8h_ppb"]
NOMBRE = {TMAX: "Temperatura máx.", "humedad_rel_media": "Humedad", "severidad_sequia": "Sequía",
          "pm10_ugm3": "PM10", "o3_max8h_ppb": "Ozono"}

# Temporadas del clima de Hermosillo (las mismas de 3.3) y su color en las gráficas
TEMPORADA_POR_MES = {11: "frío (nov–feb)", 12: "frío (nov–feb)", 1: "frío (nov–feb)", 2: "frío (nov–feb)",
                     3: "seco (mar–jun)", 4: "seco (mar–jun)", 5: "seco (mar–jun)", 6: "seco (mar–jun)",
                     7: "monzón (jul–oct)", 8: "monzón (jul–oct)", 9: "monzón (jul–oct)", 10: "monzón (jul–oct)"}
COLORES = {"frío (nov–feb)": "#2a78d6", "seco (mar–jun)": "#eb6834", "monzón (jul–oct)": "#1baf7a"}

# Lee el panel. Si no existe, se detiene y dice qué notebook hay que correr.
ruta = PROCESSED_DATA_DIR / _CFG["archivo"]
if not ruta.exists():
    raise FileNotFoundError(f"Falta {ruta.name}: completar el paso 7 de 3.0-integracion-espacial.ipynb")
panel = pd.read_parquet(ruta).sort_values(FECHA).reset_index(drop=True)

# Sin 2020 (pandemia); se agrega el mes y la temporada de cada periodo
base = panel[~panel["anio_atipico"].astype(bool)].copy()
base["mes"] = base[FECHA].dt.month
base["temporada"] = base["mes"].map(TEMPORADA_POR_MES)

# "Sin la época": cada valor menos el promedio de su mes (se calcula con TODOS los periodos
# de 2015-2019, antes de quitar los incompletos, igual que en 3.4)
columnas = VARIABLES + [Y]
sin_epoca = base[columnas] - base.groupby("mes")[columnas].transform("mean")

# PCA y t-SNE no aceptan huecos: se quedan solo los periodos con las 5 variables medidas
completos = base[VARIABLES].notna().all(axis=1)
datos = base[completos].reset_index(drop=True)
datos_sin_epoca = sin_epoca[completos].reset_index(drop=True)
print(f"Panel {GRANULARIDAD}: {len(datos)} periodos completos de {len(base)} (2015-2019).")

## 1. PCA con los datos originales: ¿cuántos ejes hacen falta?

A la izquierda, **cuánta información guarda cada eje** (el porcentaje de la
variación de las 5 variables que explica). A la derecha, los **pesos** de las
variables en los dos primeros ejes: rojo y azul fuertes = la variable define
ese eje; blanco = casi no aporta.

**Resultado:** los dos primeros ejes guardan el **66 %** de la información
(40 % y 26 %); con 4 ejes ya se llega al 95 %.

- **Eje 1, "seco contra húmedo":** de un lado la humedad (+0.62); del otro el
  ozono (−0.53) y la sequía (−0.45). Es la diferencia entre la primavera seca y
  las semanas húmedas.
- **Eje 2, "invierno con polvo contra verano caluroso":** de un lado el PM10
  (+0.75); del otro la temperatura (−0.61).

In [ ]:
# 1) Estandarizar: cada variable queda con promedio 0 y variación típica 1
X = StandardScaler().fit_transform(datos[VARIABLES])

# 2) Ajustar PCA y obtener la posición de cada periodo en los ejes nuevos
pca = PCA(random_state=42)
ejes = pca.fit_transform(X)  # una fila por periodo, una columna por eje

# Porcentaje de información que guarda cada eje
varianza = pd.Series(pca.explained_variance_ratio_ * 100, index=[f"eje {i + 1}" for i in range(len(VARIABLES))])
print("Información que guarda cada eje (%):", varianza.round(1).to_dict())
print("Acumulado (%):", varianza.cumsum().round(1).to_dict())

# Pesos de cada variable en los dos primeros ejes (pca.components_ tiene un renglón por eje)
pesos = pd.DataFrame(pca.components_[:2].T, index=[NOMBRE[v] for v in VARIABLES], columns=["eje 1", "eje 2"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
varianza.plot.bar(ax=axes[0], color="#2a78d6", rot=0)  # una barra por eje
axes[0].set_ylabel("% de la información")
axes[0].set_title("¿Cuánta información guarda cada eje?", loc="left")
sns.heatmap(pesos, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Pesos de cada variable (datos originales)", loc="left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"pca_ejes_{GRANULARIDAD}.png", dpi=150)
plt.show()

## 2. ¿Las semanas forman grupos?

Cada punto es una semana, colocada según sus dos primeros ejes. A la izquierda
el color es la **temporada**; a la derecha, los **casos de IRA** (más oscuro =
más casos).

**Resultado:** las semanas forman **tres nubes, una por temporada**. La
primavera seca (naranja) queda sola de un lado del eje 1. Las semanas de frío
(azul) quedan arriba en el eje 2, y ahí están también los **casos altos**: el
eje 2 se mueve casi igual que los casos (correlación de 0.78). Es decir, el
mapa vuelve a separar invierno de verano, igual que en 3.3 y 3.4.

In [ ]:
# Tabla con la posición de cada periodo en los dos primeros ejes, su temporada y sus casos
mapa = pd.DataFrame({"eje 1": ejes[:, 0], "eje 2": ejes[:, 1],
                     "temporada": datos["temporada"], "casos": datos[Y]})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Izquierda: color por temporada (hue = columna que da el color)
sns.scatterplot(data=mapa, x="eje 1", y="eje 2", hue="temporada", palette=COLORES,
                alpha=0.7, ax=axes[0])
axes[0].set_title("Semanas por temporada", loc="left")
# Derecha: color por número de casos (más oscuro = más casos)
puntos = axes[1].scatter(mapa["eje 1"], mapa["eje 2"], c=mapa["casos"], cmap="Blues", alpha=0.8)
fig.colorbar(puntos, ax=axes[1], label="casos de IRA")
axes[1].set_xlabel("eje 1")
axes[1].set_ylabel("eje 2")
axes[1].set_title("Semanas por casos de IRA", loc="left")
fig.suptitle("PCA con los datos originales", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"pca_mapa_{GRANULARIDAD}.png", dpi=150)
plt.show()

# ¿Qué eje se mueve junto con los casos? (correlación de Spearman, como en 3.4)
print("Correlación de los casos con el eje 1:", round(mapa["casos"].corr(mapa["eje 1"], method="spearman"), 2))
print("Correlación de los casos con el eje 2:", round(mapa["casos"].corr(mapa["eje 2"], method="spearman"), 2))

## 3. PCA sin la época del año

Se repite el PCA, pero con cada semana comparada con **lo normal para su mes**
(como en 3.4). Así ya no puede separar invierno de verano: solo puede encontrar
semanas que fueron **distintas a lo normal** para su época.

**Resultado:**

- **Eje 1 (47 %), "más calurosa, seca y con más polvo de lo normal":**
  temperatura (−0.55) y PM10 (−0.52) de un lado, humedad (+0.56) del otro. Son
  las tres variables que en 3.4 se repetían entre sí.
- **Eje 2 (21 %):** sobre todo el ozono (+0.74) contra la sequía (−0.64).
- **Las temporadas se mezclan** en una sola nube (ya no hay grupos), y **los
  ejes casi no se relacionan con los casos** (−0.12 y +0.10). No hay un tipo de
  semana fuera de lo normal que tenga más casos.

In [ ]:
# Mismo procedimiento que en el paso 1, ahora con los valores sin la época
X_sin = StandardScaler().fit_transform(datos_sin_epoca[VARIABLES])
pca_sin = PCA(random_state=42)
ejes_sin = pca_sin.fit_transform(X_sin)

print("Información que guarda cada eje, sin la época (%):",
      dict(zip([f"eje {i + 1}" for i in range(len(VARIABLES))], (pca_sin.explained_variance_ratio_ * 100).round(1).tolist())))
pesos_sin = pd.DataFrame(pca_sin.components_[:2].T, index=[NOMBRE[v] for v in VARIABLES], columns=["eje 1", "eje 2"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Izquierda: pesos de las variables en los dos ejes
sns.heatmap(pesos_sin, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Pesos de cada variable (sin la época)", loc="left")
# Derecha: las semanas, coloreadas por cuántos casos de más (rojo) o de menos (azul) tuvieron
casos_sin = datos_sin_epoca[Y]
limite = casos_sin.abs().quantile(0.95)  # escala de color simétrica alrededor de 0
puntos = axes[1].scatter(ejes_sin[:, 0], ejes_sin[:, 1], c=casos_sin, cmap="RdBu_r",
                         vmin=-limite, vmax=limite, alpha=0.8)
fig.colorbar(puntos, ax=axes[1], label="casos de más (+) o de menos (−) vs. lo normal")
axes[1].set_xlabel("eje 1")
axes[1].set_ylabel("eje 2")
axes[1].set_title("Semanas sin la época", loc="left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"pca_sin_epoca_{GRANULARIDAD}.png", dpi=150)
plt.show()

print("Correlación de los casos (sin la época) con el eje 1:", round(casos_sin.corr(pd.Series(ejes_sin[:, 0]), method="spearman"), 2))
print("Correlación de los casos (sin la época) con el eje 2:", round(casos_sin.corr(pd.Series(ejes_sin[:, 1]), method="spearman"), 2))

## 4. t-SNE: otra forma de buscar grupos

t-SNE acomoda las semanas para que las que tienen clima y aire **parecidos**
queden juntas. Se usa con los **datos originales**, para comparar con el paso 2.
Ojo: en t-SNE los ejes no significan nada; solo importa qué puntos quedan
cerca.

**Resultado:** sale lo mismo que con PCA: las semanas se acomodan en **grupos
por temporada**, y los casos altos se juntan en la zona de las semanas de frío.
La única excepción es un grupito de semanas de frío (abajo a la derecha) que
queda junto a las semanas secas: es el **invierno 2017–2018** (diciembre a
febrero), que tuvo sequía fuerte (D2–D3) y aire más seco de lo normal. Sus
casos fueron los normales de invierno (5,600–6,300 por semana), así que ese
invierno seco no tuvo más casos.

In [ ]:
# perplexity = más o menos cuántos "vecinos" mira cada punto; con pocos periodos
# (panel mensual) hay que bajarlo. random_state fija el resultado para que salga igual siempre.
tsne = TSNE(n_components=2, perplexity=30 if GRANULARIDAD == "semanal" else 10,
            init="pca", learning_rate="auto", random_state=42)
posiciones = tsne.fit_transform(X)  # X = datos originales estandarizados (paso 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Izquierda: color por temporada
sns.scatterplot(x=posiciones[:, 0], y=posiciones[:, 1], hue=datos["temporada"], palette=COLORES,
                alpha=0.7, ax=axes[0])
axes[0].set_title("t-SNE por temporada", loc="left")
# Derecha: color por casos de IRA
puntos = axes[1].scatter(posiciones[:, 0], posiciones[:, 1], c=datos[Y], cmap="Blues", alpha=0.8)
fig.colorbar(puntos, ax=axes[1], label="casos de IRA")
axes[1].set_title("t-SNE por casos de IRA", loc="left")
for ax in axes:  # en t-SNE los ejes no tienen unidades: se quitan los números
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"tsne_{GRANULARIDAD}.png", dpi=150)
plt.show()

## Conclusiones

Resultados del **panel semanal** (225 semanas completas, 2015–2019):

1. **Las 5 variables se pueden resumir en 2 ejes** que guardan el 66 % de la
   información: "seco contra húmedo" y "invierno con polvo contra verano
   caluroso".
2. **Las semanas se agrupan por temporada**, con PCA y con t-SNE, y los casos
   altos caen en el grupo del invierno. Es la misma historia de 3.3 y 3.4: la
   época del año domina todo.
3. **Sin la época, no hay grupos ni relación con los casos.** El eje principal
   ("más calurosa, seca y con más polvo de lo normal", 47 %) junta temperatura,
   humedad y PM10, pero casi no se relaciona con los casos (−0.12).
4. **Para un modelo futuro:** temperatura, humedad y PM10 se podrían resumir en
   un solo índice (el eje 1 sin la época), para no meter tres variables que
   cuentan lo mismo.

**Panel mensual** (cambiar `GRANULARIDAD = "mensual"`, 47 meses completos):
sale lo mismo. Con los datos originales, dos ejes guardan el 71 % y se separan
las temporadas; sin la época, el eje 1 vuelve a juntar temperatura, humedad y
PM10 (54 %) y no se relaciona con los casos; el eje 2, que es casi solo el
ozono, sí (+0.36), igual que el ozono en 3.4 (+0.37), con solo 47 meses.

**Lo que no se pudo ver:** la marginación es un solo valor para todo el
municipio, así que no sirve para colorear semanas. Para buscar grupos por zona
de la ciudad hacen falta los datos por AGEB (pendientes de la cartografía del
INEGI).

## Consulta con el equipo (checkpoint 2)

Registrar aquí lo que se presentó al equipo sobre los clusters o patrones de PCA/t-SNE y qué se decidió
(ver `docs/docs/protocolo_eda.md`, sección 5.3).

| Fecha | Participantes | Hallazgo o pregunta presentada | Decisión |
|-------|---------------|-------------------------------|----------|
| _TODO_ | | | |